# The neuron you have already built

MichAl Academy, units 3.1 and 3.2.

Run each cell with **Shift+Enter**.

Lesson 2.5.3 fitted logistic regression and nobody called it a neuron. This
notebook shows the arithmetic is the same, then finds the wall a single one hits
and measures how unreliably a small network gets past it.


In [ ]:
import numpy as np
import torch
from sklearn.datasets import load_breast_cancer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# Every network here is tiny, and torch defaults to one thread per physical core
# then spends its time coordinating them rather than doing arithmetic. Measured
# in this image: one 8-unit network for 2000 epochs takes 6.33s on 6 threads and
# 2.94s on one. On a real model you would never do this.
torch.set_num_threads(1)
np.set_printoptions(precision=3, suppress=True)

print("torch", torch.__version__)


## 1. The four steps, done twice

A neuron multiplies each input by a weight, adds them up, adds a bias, and
passes the result through an activation function.

`torch.nn.Linear` does the first three. Set its weights by hand and check the
arithmetic yourself.


In [ ]:
neuron = torch.nn.Linear(3, 1)

with torch.no_grad():
    neuron.weight[:] = torch.tensor([[2.0, -1.0, 0.5]])
    neuron.bias[:] = torch.tensor([-0.3])

x = torch.tensor([[1.0, 2.0, 4.0]])

by_hand = 2.0 * 1.0 + (-1.0) * 2.0 + 0.5 * 4.0 + (-0.3)
by_torch = neuron(x).item()

print(f"weighted sum by hand  : {by_hand:.4f}")
print(f"weighted sum by torch : {by_torch:.4f}")
print(f"squashed to a probability: {torch.sigmoid(neuron(x)).item():.4f}")

assert abs(by_hand - by_torch) < 1e-6


That is the entire neuron. The activation is the only step logistic regression
does not spell out in the same words, and it is the same logistic squash.

## 2. Is it really the same model?

Fit both to the breast cancer measurements and compare. If they are the same
model, they should score the same.


In [ ]:
X, y = load_breast_cancer(return_X_y=True)
CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)


def train_neuron(Xtr, ytr, epochs=2000, lr=0.1, seed=0, weight_decay=0.0):
    """One linear unit trained with binary cross-entropy. A neuron."""
    torch.manual_seed(seed)
    Xt = torch.tensor(Xtr, dtype=torch.float32)
    yt = torch.tensor(ytr, dtype=torch.float32).unsqueeze(1)
    unit = torch.nn.Linear(Xtr.shape[1], 1)
    opt = torch.optim.Adam(unit.parameters(), lr=lr, weight_decay=weight_decay)
    lossf = torch.nn.BCEWithLogitsLoss()
    for _ in range(epochs):
        opt.zero_grad()
        lossf(unit(Xt), yt).backward()
        opt.step()
    return unit


def cv_neuron(**kw):
    scores = []
    for tr, te in CV.split(X, y):
        scaler = StandardScaler().fit(X[tr])
        unit = train_neuron(scaler.transform(X[tr]), y[tr], **kw)
        with torch.no_grad():
            Xte = torch.tensor(scaler.transform(X[te]), dtype=torch.float32)
            p = torch.sigmoid(unit(Xte)).numpy().ravel()
        scores.append(((p > 0.5).astype(int) == y[te]).mean())
    return float(np.mean(scores))


sklearn_default = cross_val_score(
    make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000)), X, y, cv=CV).mean()

print(f"scikit-learn LogisticRegression : {sklearn_default:.4f}")
print(f"one PyTorch neuron              : {cv_neuron():.4f}")


Different. They are the same four operations, so the difference has to be in the
settings, and it is. Two of them.

**scikit-learn penalises large weights and does not say so.** `LogisticRegression`
applies L2 regularisation by default. The bare neuron applies none.


In [ ]:
sklearn_none = cross_val_score(
    make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000, penalty=None)),
    X, y, cv=CV).mean()

print(f"scikit-learn, penalty switched off : {sklearn_none:.4f}")
print(f"neuron, given the same kind of penalty : {cv_neuron(weight_decay=0.01):.4f}")


**And two thousand passes had not converged.** Train the neuron ten times longer
and compare its weights against the unpenalised scikit-learn fit, which is now
the like-for-like pair.


In [ ]:
scaler = StandardScaler().fit(X)
Xs = scaler.transform(X)

w_none = LogisticRegression(max_iter=5000, penalty=None).fit(Xs, y).coef_.ravel()
w_l2 = LogisticRegression(max_iter=5000).fit(Xs, y).coef_.ravel()

for epochs in (2000, 20000):
    w_nn = train_neuron(Xs, y, epochs=epochs).weight.detach().numpy().ravel()
    print(f"neuron at {epochs:5d} epochs vs scikit-learn unpenalised : "
          f"corr {np.corrcoef(w_none, w_nn)[0, 1]:.4f}")

print()
print(f"scikit-learn default vs scikit-learn unpenalised   : "
      f"corr {np.corrcoef(w_l2, w_none)[0, 1]:.4f}")


Read the last line twice. The penalty moves the weights further than swapping
the entire framework does.

That is lesson 2.4's coefficient trap again: a weight means nothing outside the
exact setup that produced it, and "the same model" is a claim about settings
rather than about libraries.

## 3. The wall

Four points. The answer is 1 when exactly one input is 1. This is XOR.


In [ ]:
XOR_X = np.array([[0., 0.], [0., 1.], [1., 0.], [1., 1.]])
XOR_y = np.array([0, 1, 1, 0])

for row, label in zip(XOR_X, XOR_y):
    print(f"  ({row[0]:.0f}, {row[1]:.0f})  ->  {label}")


A neuron draws one straight boundary. Rather than argue about whether that is
enough, check every boundary: a 0.1 grid from -4 to 4 on both weights and the
bias, which is 531,441 of them.

A point sitting exactly on the line is not a decision, so require every point to
sit clear of it.


In [ ]:
grid = np.linspace(-4, 4, 81)
W0, W1, B = np.meshgrid(grid, grid, grid, indexing="ij")
params = np.stack([W0.ravel(), W1.ravel(), B.ravel()], axis=1)

Xaug = np.hstack([XOR_X, np.ones((4, 1))])
Z = Xaug @ params.T

valid = np.abs(Z).min(axis=0) >= 1e-3
correct = ((Z > 0).astype(int) == XOR_y[:, None]).sum(axis=0)

best = correct[valid].max()
print(f"boundaries checked : {params.shape[0]}")
print(f"best any of them does : {best} of 4")
print(f"how many reach it     : {(valid & (correct == best)).sum()}")


Three of four, and never four. The tidy version of that ceiling is `OR`.


In [ ]:
w, b = np.array([1.0, 1.0]), -0.5
z = XOR_X @ w + b

print(f"weights {w}, bias {b}")
print(f"weighted sums : {z}")
print(f"answers       : {(z > 0).astype(int)}")
print(f"XOR wanted    : {XOR_y}")
print(f"correct       : {int(((z > 0).astype(int) == XOR_y).sum())} of 4")


The sharper way to say what is missing. There are 16 possible ways to label four
points. Count how many a straight boundary can actually produce.


In [ ]:
reachable = set(map(tuple, (Z[:, valid] > 0).astype(int).T.tolist()))
everything = [tuple((i >> k) & 1 for k in (3, 2, 1, 0)) for i in range(16)]

print(f"labellings a straight boundary can produce : {len(reachable)} of 16")
print(f"the ones it cannot : {[p for p in everything if p not in reachable]}")


Exactly two are out of reach: XOR, and XOR upside down.

## 4. It is worse than the ceiling

A neuron cannot express XOR. Train one anyway and watch it fail to reach even
the 3 of 4 it could have expressed.


In [ ]:
unit = train_neuron(XOR_X, XOR_y.astype(float), epochs=5000, lr=0.05)

with torch.no_grad():
    probs = torch.sigmoid(unit(torch.tensor(XOR_X, dtype=torch.float32))).numpy().ravel()

print(f"weights : {unit.weight.detach().numpy().ravel()}")
print(f"bias    : {unit.bias.detach().numpy()}")
print(f"outputs : {probs}")
print(f"correct : {int(((probs > 0.5).astype(int) == XOR_y).sum())} of 4")


Every weight is zero and every output is exactly one half, which scores 2 of 4.
A coin flip.

XOR is symmetric: swap the two inputs and the answers do not change. Every pull
gradient descent feels on a weight is matched by an equal pull the other way, so
they cancel to nothing.

Two separate failures, worth keeping apart. The model **cannot represent** the
answer. Gradient descent **cannot find** the best answer the model could have
represented. Lesson 3.3 is largely about the second kind.

## 5. Two units, twenty times

Two neurons in a hidden layer are enough to represent XOR. Whether training
finds it is a different question, so run it twenty times from twenty starts.

This is the slow cell in the notebook.


In [ ]:
def train_xor_net(hidden, act, seed, epochs=2000, lr=0.1):
    torch.manual_seed(seed)
    Xt = torch.tensor(XOR_X, dtype=torch.float32)
    yt = torch.tensor(XOR_y, dtype=torch.float32).unsqueeze(1)
    activation = {"relu": torch.nn.ReLU, "tanh": torch.nn.Tanh}[act]
    net = torch.nn.Sequential(
        torch.nn.Linear(2, hidden), activation(), torch.nn.Linear(hidden, 1))
    opt = torch.optim.Adam(net.parameters(), lr=lr)
    lossf = torch.nn.BCEWithLogitsLoss()
    for _ in range(epochs):
        opt.zero_grad()
        lossf(net(Xt), yt).backward()
        opt.step()
    with torch.no_grad():
        probs = torch.sigmoid(net(Xt)).numpy().ravel()
        hidden_out = net[1](net[0](Xt)).numpy()
    solved = bool(((probs > 0.5).astype(int) == XOR_y).all())
    all_dead = bool(np.abs(hidden_out).max() < 1e-6)
    return solved, all_dead


SEEDS = 20

# Train every combination once and keep both facts about each run, so the next
# question does not cost another sweep.
runs = {(act, hidden): [train_xor_net(hidden, act, seed) for seed in range(SEEDS)]
        for act in ("relu", "tanh") for hidden in (2, 4, 8)}

print(f"{'':10}{'relu':>10}{'tanh':>10}")
for hidden in (2, 4, 8):
    counts = [sum(1 for solved, _ in runs[(act, hidden)] if solved)
              for act in ("relu", "tanh")]
    print(f"{hidden:>3} units {counts[0]:>6} /{SEEDS:<3}{counts[1]:>6} /{SEEDS:<3}")


Two units are sufficient in principle and unreliable in practice.

It is tempting to blame ReLU. A ReLU unit that goes negative for every input
passes no gradient and stays dead. That does happen here, so count it, using the
runs already done.


In [ ]:
for act in ("relu", "tanh"):
    dead = sum(1 for _, all_dead in runs[(act, 2)] if all_dead)
    failed = sum(1 for solved, _ in runs[(act, 2)] if not solved)
    print(f"{act:5} at 2 units: {failed} failures, of which {dead} had every unit dead")


Dead units explain some of the ReLU failures and none of the tanh ones, yet tanh
still fails most of the time at two units. So the activation function is not the
story.

**Extra width is buying attempts, not capacity.** Four units are no more
expressive than two for this problem. They just give the descent more places to
start from where it can get somewhere.

## 6. What you have

- A neuron is logistic regression with different words for the same four steps.
- "Same model" is a claim about settings. Match the penalty and the training
  length or the comparison is meaningless.
- One neuron produces 14 of the 16 labellings of four points. XOR and its mirror
  image are the two it cannot.
- Being able to represent an answer and being able to find it are different
  problems.

Unit 3.3 shows how the error at the output gets shared backwards among weights
that never saw it.


## 7. The activation is not optional

Atom 3.1.2 claims a step activation hands training nothing to work with, and
atom 3.2.3 claims two linear layers with no activation are one linear layer.
Both are checkable here.


In [ ]:
w = torch.tensor([0.5], requires_grad=True)
x = torch.tensor([2.0])

step_out = (w * x > 0).float()          # a step activation
print("step output:", step_out.item(), " requires_grad:", step_out.requires_grad)
try:
    ((step_out - 1.0) ** 2).backward()
    print("w.grad:", w.grad)
except RuntimeError as e:
    print("backward through a step ->", e)

w2 = torch.tensor([0.5], requires_grad=True)
((torch.sigmoid(w2 * x) - 1.0) ** 2).backward()
print()
print("same thing with a logistic activation, w.grad =", w2.grad.item())


The logistic function's slope is largest at zero and falls away on both sides.
That is why it works, and later it is also why it stops working through many
layers.


In [ ]:
for z in (-2.0, -1.0, -0.5, 0.0, 0.5, 1.0, 2.0):
    s = torch.sigmoid(torch.tensor(z))
    print(f"sigmoid({z:+.1f}) = {s.item():.4f}   slope = {(s * (1 - s)).item():.4f}")


Now the collapse. Multiply the two weight matrices together by hand and compare
against what the stacked layers actually produce.


In [ ]:
torch.manual_seed(0)
l1 = torch.nn.Linear(2, 8)
l2 = torch.nn.Linear(8, 1)

XOR_X = torch.tensor([[0., 0.], [0., 1.], [1., 0.], [1., 1.]])
XOR_Y = torch.tensor([[0.], [1.], [1.], [0.]])

stacked = l2(l1(XOR_X))
W = l2.weight @ l1.weight                 # one 1x2 matrix
b = l2.weight @ l1.bias + l2.bias         # one bias
single = XOR_X @ W.T + b

print("stacked two layers :", stacked.detach().flatten().numpy())
print("one layer, W2 @ W1 :", single.detach().flatten().numpy())
print("largest difference :", (stacked - single).abs().max().item())
print()
print("effective weights  :", W.detach().flatten().numpy(),
      " bias", b.detach().flatten().numpy())


Eight hidden units, two effective weights. So a linear-only stack meets XOR with
the ceiling a single neuron already had. Twenty starts each way.


In [ ]:
def train_xor(make_model, seeds=20, epochs=1500, lr=0.05):
    wins, best = 0, 0
    for seed in range(seeds):
        torch.manual_seed(seed)
        model = make_model()
        opt = torch.optim.Adam(model.parameters(), lr=lr)
        lossf = torch.nn.BCEWithLogitsLoss()
        for _ in range(epochs):
            opt.zero_grad()
            lossf(model(XOR_X), XOR_Y).backward()
            opt.step()
        with torch.no_grad():
            correct = int((((model(XOR_X) > 0).float()) == XOR_Y).sum())
        best = max(best, correct)
        wins += correct == 4
    return wins, best


for name, maker in [
    ("2-8-1, no activation", lambda: torch.nn.Sequential(
        torch.nn.Linear(2, 8), torch.nn.Linear(8, 1))),
    ("2-8-1, tanh", lambda: torch.nn.Sequential(
        torch.nn.Linear(2, 8), torch.nn.Tanh(), torch.nn.Linear(8, 1))),
]:
    wins, best = train_xor(maker)
    print(f"{name:22s} solved {wins}/20, best {best} of 4")


## 8. What the hidden layer actually did

Train the smallest network that works, then look at where the four points ended
up after the hidden layer but before the output neuron.


In [ ]:
torch.manual_seed(3)
net = torch.nn.Sequential(torch.nn.Linear(2, 2), torch.nn.Tanh(),
                          torch.nn.Linear(2, 1))
opt = torch.optim.Adam(net.parameters(), lr=0.05)
lossf = torch.nn.BCEWithLogitsLoss()
for _ in range(4000):
    opt.zero_grad()
    lossf(net(XOR_X), XOR_Y).backward()
    opt.step()

with torch.no_grad():
    hidden = torch.tanh(net[0](XOR_X))
    correct = int((((net(XOR_X) > 0).float()) == XOR_Y).sum())

print(f"solved {correct} of 4")
print()
print("input     XOR   after the hidden layer")
for xi, yi, hi in zip(XOR_X, XOR_Y, hidden):
    print(f"{tuple(xi.tolist())}    {int(yi.item())}     "
          f"({hi[0]:+.2f}, {hi[1]:+.2f})")


The two points that should answer 0 have been moved onto the same spot, and the
two that should answer 1 have been pushed to different corners. Four points
became three positions, and a straight line separates those.

The hidden layer did not learn XOR. It made XOR linearly separable, and then one
neuron was enough.
